# Fintech RAG — Exploration Notebook

Use this notebook to prototype ingestion, retrieval, and generation before wiring into the API.

In [ ]:
import sys
sys.path.insert(0, '..')

from dotenv import load_dotenv
load_dotenv('../.env')

## 1. Load and chunk documents

In [ ]:
from pathlib import Path
from src.ingestion.loaders import load_directory
from src.ingestion.chunking import chunk_documents, ChunkStrategy

docs = load_directory(Path('../data/raw'))
print(f'Loaded {len(docs)} documents')

chunks = chunk_documents(docs, strategy=ChunkStrategy.RECURSIVE)
print(f'Created {len(chunks)} chunks')
print('\nSample chunk:')
print(chunks[0].page_content[:300] if chunks else 'No chunks')

## 2. Fetch SEC filings from EDGAR

In [ ]:
from src.ingestion.loaders import iter_recent_filings

for filing in iter_recent_filings('AAPL', form_type='10-K', limit=3):
    print(filing)

## 3. Ingest into Qdrant

In [ ]:
# Requires Qdrant running: docker run -p 6333:6333 qdrant/qdrant
from src.ingestion.pipeline import ingest_local
n = ingest_local(Path('../data/raw'))
print(f'Upserted {n} chunks')

## 4. Hybrid search

In [ ]:
from src.retrieval.hybrid_search import hybrid_search

results = hybrid_search('What was the gross margin trend over the last three years?')
for i, doc in enumerate(results):
    print(f'[{i+1}] {doc.metadata.get("filename", "?")} chunk {doc.metadata.get("chunk_index", "?")}:')
    print(doc.page_content[:200])
    print()

## 5. Full RAG query

In [ ]:
from src.generation.chain import rag_query

result = rag_query(
    'What are the key risk factors mentioned in the most recent 10-K?',
    retrieval_mode='hybrid',
)
print(result['answer'])
print('\nSources:')
for src in result['sources']:
    print(' -', src.get('filename'), 'page', src.get('page'))